# AgriSmart: Dataset Preparation

This notebook cleans and combines the three source datasets (APMC, India Agriculture, and ICRISAT). It creates `datasets/processed/crop_data_v2.csv`, which is used by the model-training notebook.

**Run this notebook first.** Keep the `datasets` folder in the project root and open Jupyter with `AgriSmart_Corrected` as the working directory.

## 1. Install required libraries

Run this cell once if the libraries are not already installed.

In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load, clean, merge, and save the datasets

The following cell performs the same preprocessing and matching steps as the original `prepare_dataset.py`.

In [2]:
"""
prepare_dataset.py
==================
Merges the three source datasets into crop_data_v2.csv:

  Dataset 1  (APMC)                 → primary grain: one row per market-price observation
  Dataset 2  (India Agriculture)    → Area, Production, Yield joined on State+District+Crop+Season
  Dataset 3  (ICRISAT, unpivoted)   → fallback Area/Production/Yield when Dataset 2 misses

Run from the project root or code folder:
    python code/prepare_dataset.py
"""

import os
import re

import numpy as np
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────────────────
from pathlib import Path

# Set the project folder
PROJECT_DIR = Path.cwd()
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "datasets" / "raw").exists():
    candidates = [p for p in PROJECT_DIR.parents if (p / "datasets" / "raw").exists()]
    if not candidates:
        raise FileNotFoundError(
            "Could not find datasets/raw. Open Jupyter from the AgriSmart_Corrected project folder."
        )
    PROJECT_DIR = candidates[0]
RAW_DIR = PROJECT_DIR / "datasets" / "raw"
PROCESSED_DIR = PROJECT_DIR / "datasets" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

APMC_CSV = RAW_DIR / "APMC_Market_Prices.csv"
INDIA_CSV = RAW_DIR / "India_Agriculture_Crop_Production.csv"
ICRISAT_CSV = RAW_DIR / "ICRISAT_District_Level_Data.csv"
OUT_CSV = PROCESSED_DIR / "crop_data_v2.csv"


# ── Helpers ───────────────────────────────────────────────────────────────────
def norm(s):
    """Lowercase + strip + remove parenthetical variants for fuzzy matching."""
    s = str(s).strip()
    s = re.sub(r"\s*\(.*?\)", "", s)
    return s.strip().lower()


CROP_ALIASES = {
    "cucumbar": "cucumber",
    "indian beans": "beans",
    "castor seed": "castor",
    "rapeseed &mustard": "mustard",
    "rapeseed and mustard": "mustard",
    "dry chillies": "chillies",
}


def norm_crop(s):
    value = norm(s)
    value = value.replace("&", "and")
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return CROP_ALIASES.get(value, value)


def month_to_season(m: int) -> str:
    if m in (6, 7, 8, 9, 10):
        return "Kharif"
    elif m in (11, 12, 1, 2, 3):
        return "Rabi"
    return "Zaid"


# ══════════════════════════════════════════════════════════════════════════════
# 1. APMC — primary grain
# ══════════════════════════════════════════════════════════════════════════════
print("Loading APMC dataset...")
apmc = pd.read_csv(APMC_CSV)
apmc.columns = apmc.columns.str.strip()

# URL-encoded spaces in column names
apmc.rename(columns={
    "Min_x0020_Price":   "Min_Price",
    "Max_x0020_Price":   "Max_Price",
    "Modal_x0020_Price": "Modal_Price",
}, inplace=True)

apmc["Arrival_Date"] = pd.to_datetime(
    apmc["Arrival_Date"], dayfirst=True, errors="coerce"
)
apmc.dropna(subset=["Arrival_Date", "Modal_Price"], inplace=True)
apmc = apmc[apmc["Modal_Price"] > 0]

apmc["Year"]   = apmc["Arrival_Date"].dt.year
apmc["Month"]  = apmc["Arrival_Date"].dt.month
apmc["Season"] = apmc["Month"].map(month_to_season)

# Normalised keys for joining
apmc["_k_state"]    = apmc["State"].map(norm)
apmc["_k_district"] = apmc["District"].map(norm)
apmc["_k_crop"]     = apmc["Commodity"].map(norm_crop)
apmc["_k_season"]   = apmc["Season"].map(norm)

print(f"   {len(apmc):,} rows after cleaning.")


# ══════════════════════════════════════════════════════════════════════════════
# 2. India Agriculture — Area / Production / Yield by State+District+Crop+Season
# ══════════════════════════════════════════════════════════════════════════════
print("Loading India Agriculture dataset...")
india = pd.read_csv(INDIA_CSV)
india.columns = india.columns.str.strip()

# Year column is "2001-02" → keep the start year as int
india["Year"] = india["Year"].astype(str).str.split("-").str[0]
india["Year"] = pd.to_numeric(india["Year"], errors="coerce")

india["Crop"]     = india["Crop"].str.strip()
india["State"]    = india["State"].str.strip()
india["District"] = india["District"].str.strip().str.title()
india["Season"]   = india["Season"].str.strip()
india["Area"]     = pd.to_numeric(india["Area"],       errors="coerce")
india["Production"]= pd.to_numeric(india["Production"],errors="coerce")
india["Yield"]    = pd.to_numeric(india["Yield"],      errors="coerce")

# Aggregate to State+District+Crop+Season (average over all years)
india_agg = (
    india.groupby(["State", "District", "Crop", "Season"], as_index=False)
    .agg(Area=("Area", "mean"), Production=("Production", "mean"), Yield=("Yield", "mean"))
)
india_agg["_k_state"]    = india_agg["State"].map(norm)
india_agg["_k_district"] = india_agg["District"].map(norm)
india_agg["_k_crop"]     = india_agg["Crop"].map(norm_crop)
india_agg["_k_season"]   = india_agg["Season"].map(norm)

india_no_season = (
    india.groupby(["State", "District", "Crop"], as_index=False)
    .agg(Area_NS=("Area", "mean"), Production_NS=("Production", "mean"), Yield_NS=("Yield", "mean"))
)
india_no_season["_k_state"] = india_no_season["State"].map(norm)
india_no_season["_k_district"] = india_no_season["District"].map(norm)
india_no_season["_k_crop"] = india_no_season["Crop"].map(norm_crop)

india_state_crop = (
    india.groupby(["State", "Crop"], as_index=False)
    .agg(Area_SC=("Area", "mean"), Production_SC=("Production", "mean"), Yield_SC=("Yield", "mean"))
)
india_state_crop["_k_state"] = india_state_crop["State"].map(norm)
india_state_crop["_k_crop"] = india_state_crop["Crop"].map(norm_crop)

print(f"   {len(india_agg):,} State+District+Crop+Season combinations.")


# ══════════════════════════════════════════════════════════════════════════════
# 3. ICRISAT — unpivot wide → long, use as fallback
# ══════════════════════════════════════════════════════════════════════════════
print("Loading ICRISAT dataset...")
icrisat_raw = pd.read_csv(ICRISAT_CSV)
icrisat_raw.columns = icrisat_raw.columns.str.strip()
icrisat_raw.rename(columns={"State Name": "State", "Dist Name": "District"}, inplace=True)

# Parse wide column headers: "RICE AREA (1000 ha)" → crop=Rice, metric=Area
id_cols = ["State", "District", "Year"]
crop_metrics: dict[str, dict] = {}
for col in icrisat_raw.columns:
    m = re.match(r"^(.*?)\s+(AREA|PRODUCTION|YIELD)\b", col)
    if m:
        crop_key = m.group(1).strip()
        metric   = m.group(2).capitalize()          # Area | Production | Yield
        crop_metrics.setdefault(crop_key, {})[metric] = col

frames = []
for crop_key, metrics in crop_metrics.items():
    if not all(k in metrics for k in ("Area", "Production", "Yield")):
        continue
    sub = icrisat_raw[id_cols + [metrics["Area"], metrics["Production"], metrics["Yield"]]].copy()
    sub.columns = ["State", "District", "Year", "Area", "Production", "Yield"]
    sub["Crop"] = crop_key.title()
    frames.append(sub)

icrisat_long = pd.concat(frames, ignore_index=True)
for col in ("Area", "Production", "Yield"):
    icrisat_long[col] = pd.to_numeric(icrisat_long[col], errors="coerce")

# Aggregate to State+District+Crop (mean across all years; ICRISAT has no season)
icrisat_agg = (
    icrisat_long.groupby(["State", "District", "Crop"], as_index=False)
    .agg(Area_I=("Area", "mean"), Production_I=("Production", "mean"), Yield_I=("Yield", "mean"))
)
icrisat_agg["_k_state"]    = icrisat_agg["State"].map(norm)
icrisat_agg["_k_district"] = icrisat_agg["District"].map(norm)
icrisat_agg["_k_crop"]     = icrisat_agg["Crop"].map(norm_crop)

icrisat_state_crop = (
    icrisat_long.groupby(["State", "Crop"], as_index=False)
    .agg(Area_IS=("Area", "mean"), Production_IS=("Production", "mean"), Yield_IS=("Yield", "mean"))
)
icrisat_state_crop["_k_state"] = icrisat_state_crop["State"].map(norm)
icrisat_state_crop["_k_crop"] = icrisat_state_crop["Crop"].map(norm_crop)

print(f"   {len(icrisat_long):,} long-form rows -> {len(icrisat_agg):,} aggregated.")


# ══════════════════════════════════════════════════════════════════════════════
# 4. Join: APMC ← India Agriculture (primary)
# ══════════════════════════════════════════════════════════════════════════════
print("Joining APMC <- India Agriculture...")
merged = apmc.merge(
    india_agg[["_k_state", "_k_district", "_k_crop", "_k_season", "Area", "Production", "Yield"]],
    on=["_k_state", "_k_district", "_k_crop", "_k_season"],
    how="left",
)
hits_india = merged["Area"].notna().sum()
print(f"   India Agriculture join: {hits_india:,} / {len(merged):,} rows matched.")

missing = merged["Area"].isna()
hits_no_season = 0
if missing.sum() > 0:
    fb = merged.loc[missing, ["_k_state", "_k_district", "_k_crop"]].merge(
        india_no_season[["_k_state", "_k_district", "_k_crop", "Area_NS", "Production_NS", "Yield_NS"]],
        on=["_k_state", "_k_district", "_k_crop"],
        how="left",
    )
    merged.loc[missing, "Area"] = fb["Area_NS"].values
    merged.loc[missing, "Production"] = fb["Production_NS"].values
    merged.loc[missing, "Yield"] = fb["Yield_NS"].values
    hits_no_season = merged["Area"].notna().sum() - hits_india
    print(f"   India no-season fallback: {hits_no_season:,} additional rows matched.")

missing = merged["Area"].isna()
hits_state_crop = 0
if missing.sum() > 0:
    fb = merged.loc[missing, ["_k_state", "_k_crop"]].merge(
        india_state_crop[["_k_state", "_k_crop", "Area_SC", "Production_SC", "Yield_SC"]],
        on=["_k_state", "_k_crop"],
        how="left",
    )
    merged.loc[missing, "Area"] = fb["Area_SC"].values
    merged.loc[missing, "Production"] = fb["Production_SC"].values
    merged.loc[missing, "Yield"] = fb["Yield_SC"].values
    hits_state_crop = merged["Area"].notna().sum() - hits_india - hits_no_season
    print(f"   India state+crop fallback: {hits_state_crop:,} additional rows matched.")


# ══════════════════════════════════════════════════════════════════════════════
# 5. Fallback: ICRISAT for rows India Agriculture missed
# ══════════════════════════════════════════════════════════════════════════════
print("ICRISAT fallback enrichment...")
missing = merged["Area"].isna()
if missing.sum() > 0:
    fb = merged.loc[missing, ["_k_state", "_k_district", "_k_crop"]].merge(
        icrisat_agg[["_k_state", "_k_district", "_k_crop", "Area_I", "Production_I", "Yield_I"]],
        on=["_k_state", "_k_district", "_k_crop"],
        how="left",
    )
    merged.loc[missing, "Area"]       = fb["Area_I"].values
    merged.loc[missing, "Production"] = fb["Production_I"].values
    merged.loc[missing, "Yield"]      = fb["Yield_I"].values
    hits_icrisat = merged.loc[missing, "Area"].notna().sum()
    print(f"   ICRISAT fallback: {hits_icrisat:,} / {missing.sum():,} remaining rows matched.")
else:
    print("   No missing rows - ICRISAT fallback not needed.")

missing = merged["Area"].isna()
if missing.sum() > 0:
    fb = merged.loc[missing, ["_k_state", "_k_crop"]].merge(
        icrisat_state_crop[["_k_state", "_k_crop", "Area_IS", "Production_IS", "Yield_IS"]],
        on=["_k_state", "_k_crop"],
        how="left",
    )
    merged.loc[missing, "Area"] = fb["Area_IS"].values
    merged.loc[missing, "Production"] = fb["Production_IS"].values
    merged.loc[missing, "Yield"] = fb["Yield_IS"].values
    hits_icrisat_state = merged.loc[missing, "Area"].notna().sum()
    print(f"   ICRISAT state+crop fallback: {hits_icrisat_state:,} / {missing.sum():,} remaining rows matched.")


# ══════════════════════════════════════════════════════════════════════════════
# 6. Final output
# ══════════════════════════════════════════════════════════════════════════════
keep = [
    "Commodity", "Variety", "Grade",
    "State", "District", "Market",
    "Year", "Season",
    "Area", "Production", "Yield",
    "Min_Price", "Max_Price", "Modal_Price",
]
df_final = merged[keep].copy()
df_final.dropna(subset=["Modal_Price"], inplace=True)
df_final = df_final[df_final["Modal_Price"] > 0]

print("\nFinal dataset")
print(f"   Shape          : {df_final.shape}")
print(f"   Unique crops   : {df_final['Commodity'].nunique()}")
print(f"   Unique states  : {df_final['State'].nunique()}")
print(f"   Unique seasons : {df_final['Season'].nunique()}")
print(f"   Rows w/ Area   : {df_final['Area'].notna().sum():,}")
print(f"   Price range    : Rs.{df_final['Modal_Price'].min():.0f} - Rs.{df_final['Modal_Price'].max():.0f}")
print(f"\n   Sample:")
print(df_final.head(5).to_string(index=False))

df_final.to_csv(OUT_CSV, index=False)
print(f"\nSaved -> {OUT_CSV}")


Loading APMC dataset...
   2,444 rows after cleaning.
Loading India Agriculture dataset...
   35,573 State+District+Crop+Season combinations.
Loading ICRISAT dataset...
   285,614 long-form rows -> 7,153 aggregated.
Joining APMC <- India Agriculture...
   India Agriculture join: 0 / 2,444 rows matched.
   India no-season fallback: 358 additional rows matched.
   India state+crop fallback: 165 additional rows matched.
ICRISAT fallback enrichment...
   ICRISAT fallback: 10 / 1,921 remaining rows matched.
   ICRISAT state+crop fallback: 10 / 1,911 remaining rows matched.

Final dataset
   Shape          : (2444, 14)
   Unique crops   : 151
   Unique states  : 21
   Unique seasons : 1
   Rows w/ Area   : 543
   Price range    : Rs.30 - Rs.80000

   Sample:
         Commodity Variety   Grade             State District         Market  Year Season      Area  Production    Yield  Min_Price  Max_Price  Modal_Price
           Brinjal   Other Grade B            Odisha Bolangir Kantabaji APMC  202